# 주간 트렌드 리포트
셀 순서대로 실행하세요. **1번 셀만 본인 정보로 수정**하면 됩니다.

In [ ]:
# ✏️ ── 여기만 수정하세요 ─────────────────────────────────

CLAUDE_API_KEY      = "여기에_Claude_API_키"
NAVER_CLIENT_ID     = "여기에_네이버_Client_ID"
NAVER_CLIENT_SECRET = "여기에_네이버_Client_Secret"

START_DATE = "2026-04-14"   # 분석 시작일
END_DATE   = "2026-04-20"   # 분석 종료일

# 상품 데이터 (Redash에서 복붙하거나 직접 입력)
PRODUCTS = [
    {"product_name": "청춘 앨범 A ver.",  "artist_name": "아티스트A", "visitor_cnt": 12400, "buyer_cnt": 1820, "gmv": 54600000},
    {"product_name": "굿즈 키링 세트",    "artist_name": "아티스트A", "visitor_cnt":  8300, "buyer_cnt":  960, "gmv": 28800000},
    {"product_name": "포토카드 랜덤팩",   "artist_name": "아티스트B", "visitor_cnt":  7200, "buyer_cnt":  830, "gmv": 16600000},
    {"product_name": "후드티 블랙",       "artist_name": "아티스트B", "visitor_cnt":  5100, "buyer_cnt":  420, "gmv": 37800000},
    {"product_name": "머그컵 한정판",     "artist_name": "아티스트C", "visitor_cnt":  4800, "buyer_cnt":  380, "gmv": 11400000},
]

TOP_N = 5  # 분석할 상품 수

print("설정 완료")

In [ ]:
# ── 라이브러리 & 함수 정의 (수정 불필요) ────────────────────

import requests, re, json
from datetime import datetime, timedelta

# 방문자·구매자 가중점수로 TOP N 정렬
def get_top_products(products, top_n):
    for p in products:
        p["score"] = p["visitor_cnt"] * 0.4 + p["buyer_cnt"] * 0.6
    return sorted(products, key=lambda x: x["score"], reverse=True)[:top_n]

# 네이버 DataLab: 검색량 트렌드
def get_search_trend(keyword, start, end):
    prev_start = (datetime.strptime(start, "%Y-%m-%d") - timedelta(days=7)).strftime("%Y-%m-%d")
    prev_end   = (datetime.strptime(end,   "%Y-%m-%d") - timedelta(days=7)).strftime("%Y-%m-%d")
    try:
        resp = requests.post(
            "https://openapi.naver.com/v1/datalab/search",
            headers={"X-Naver-Client-Id": NAVER_CLIENT_ID,
                     "X-Naver-Client-Secret": NAVER_CLIENT_SECRET,
                     "Content-Type": "application/json"},
            json={"startDate": prev_start, "endDate": end, "timeUnit": "date",
                  "keywordGroups": [{"groupName": keyword, "keywords": [keyword]}]},
            timeout=10
        )
        data_list = resp.json().get("results", [{}])[0].get("data", [])
        this_w = [d["ratio"] for d in data_list if start  <= d["period"] <= end]
        prev_w = [d["ratio"] for d in data_list if prev_start <= d["period"] <= prev_end]
        this_avg = round(sum(this_w)/len(this_w), 1) if this_w else 0
        prev_avg = round(sum(prev_w)/len(prev_w), 1) if prev_w else 0
        chg = round((this_avg - prev_avg) / prev_avg * 100, 1) if prev_avg else 0
        return {"this_avg": this_avg, "prev_avg": prev_avg, "change_pct": chg,
                "daily": [{"date": d["period"], "ratio": d["ratio"]} for d in data_list if start <= d["period"] <= end]}
    except Exception as e:
        print("  DataLab 오류 ({}): {}".format(keyword, e))
        return None

# 네이버 뉴스 검색
def get_news(keyword, start, end):
    TAG = re.compile(r"<[^>]+>")
    def clean(t): return TAG.sub("", t).replace("&quot;", '"').replace("&amp;", "&").strip()
    def parse_date(pub):
        try: return datetime.strptime(pub[:16], "%a, %d %b %Y").strftime("%Y-%m-%d")
        except: return ""
    try:
        resp = requests.get(
            "https://openapi.naver.com/v1/search/news.json",
            headers={"X-Naver-Client-Id": NAVER_CLIENT_ID, "X-Naver-Client-Secret": NAVER_CLIENT_SECRET},
            params={"query": keyword, "display": 20, "sort": "date"},
            timeout=10
        )
        items = resp.json().get("items", [])
        filtered = []
        for item in items:
            pub_date = parse_date(item.get("pubDate", ""))
            if pub_date and start <= pub_date <= end:
                filtered.append({"title": clean(item.get("title","")),
                                 "link":  item.get("originallink") or item.get("link",""),
                                 "description": clean(item.get("description","")),
                                 "pubDate": pub_date})
            if len(filtered) >= 5: break
        if not filtered:
            for item in items[:5]:
                filtered.append({"title": clean(item.get("title","")),
                                 "link":  item.get("originallink") or item.get("link",""),
                                 "description": clean(item.get("description","")),
                                 "pubDate": parse_date(item.get("pubDate",""))})
        return filtered
    except Exception as e:
        print("  뉴스 오류 ({}): {}".format(keyword, e))
        return []

# 카페글 검색
def get_cafe(keyword, start, end):
    TAG = re.compile(r"<[^>]+>")
    def clean(t): return TAG.sub("", t).replace("&quot;", '"').replace("&amp;", "&").strip()
    def parse_date(pub):
        try: return datetime.strptime(pub[:16], "%a, %d %b %Y").strftime("%Y-%m-%d")
        except: return ""
    try:
        resp = requests.get(
            "https://openapi.naver.com/v1/search/cafearticle.json",
            headers={"X-Naver-Client-Id": NAVER_CLIENT_ID, "X-Naver-Client-Secret": NAVER_CLIENT_SECRET},
            params={"query": keyword, "display": 20, "sort": "date"},
            timeout=10
        )
        items = resp.json().get("items", [])
        filtered = []
        for item in items:
            pub_date = parse_date(item.get("postdate", "") or item.get("pubDate", ""))
            filtered.append({"title": clean(item.get("title","")),
                             "link":  item.get("link",""),
                             "cafeName": item.get("cafename",""),
                             "pubDate": pub_date})
            if len(filtered) >= 5: break
        return filtered
    except Exception as e:
        print("  카페 오류 ({}): {}".format(keyword, e))
        return []

# Claude API 분석
def analyze(product, trend, news, cafe):
    conv_rate = round(product["buyer_cnt"] / product["visitor_cnt"] * 100, 1) if product["visitor_cnt"] > 0 else 0
    trend_text = "검색 트렌드 데이터 없음"
    if trend:
        direction = "상승" if trend["change_pct"] > 5 else "하락" if trend["change_pct"] < -5 else "보합"
        trend_text = "네이버 검색량: 이번주 {} / 전주 {} ({} {}%)".format(
            trend["this_avg"], trend["prev_avg"], direction, abs(trend["change_pct"]))
    news_text  = "\n".join(["- [{}] {}: {}".format(n["pubDate"], n["title"], n["description"]) for n in news])  or "없음"
    cafe_text  = "\n".join(["- [{}] {} ({})".format(c["pubDate"], c["title"], c["cafeName"]) for c in cafe]) or "없음"

    prompt = (
        "상품의 주간 외부 트렌드를 분석해주세요.\n\n"
        "상품명: {}\n아티스트: {}\n기간: {} ~ {}\n"
        "방문자: {}명 / 구매자: {}명 / 전환율: {}% / 거래액: {}원\n\n"
        "[외부 트렌드]\n{}\n\n[관련 뉴스 {}건]\n{}\n\n[팬카페 반응 {}건]\n{}\n\n"
        "아래 3가지를 간결하게 작성:\n"
        "1. **외부 트렌드 요약**\n2. **내외부 연관성**\n3. **다음 주 전망** (한 줄)\n"
        "전체 200자 이내, **bold** 표시 유지"
    ).format(
        product["product_name"], product["artist_name"],
        START_DATE, END_DATE,
        "{:,}".format(product["visitor_cnt"]),
        "{:,}".format(product["buyer_cnt"]),
        conv_rate,
        "{:,}".format(product["gmv"]),
        trend_text, len(news), news_text, len(cafe), cafe_text
    )
    try:
        resp = requests.post(
            "https://api.anthropic.com/v1/messages",
            headers={"x-api-key": CLAUDE_API_KEY,
                     "anthropic-version": "2023-06-01",
                     "content-type": "application/json"},
            json={"model": "claude-sonnet-4-5", "max_tokens": 600,
                  "messages": [{"role": "user", "content": prompt}]},
            timeout=30
        )
        return resp.json()["content"][0]["text"]
    except Exception as e:
        print("  Claude 오류:", e)
        return "AI 분석 실패"

print("함수 정의 완료")

In [ ]:
# ── 실행 (자동으로 돌아갑니다) ───────────────────────────────

top_products = get_top_products(PRODUCTS, TOP_N)
print("분석 대상 상품:")
for i, p in enumerate(top_products, 1):
    print("  {}. {} ({})".format(i, p["product_name"], p["artist_name"]))

results = []
for i, product in enumerate(top_products, 1):
    print("\n[{}/{}] {} 분석 중...".format(i, len(top_products), product["product_name"]))

    trend = get_search_trend(product["product_name"], START_DATE, END_DATE)
    print("  검색 트렌드:", "이번주 {} / 전주 {}".format(trend["this_avg"], trend["prev_avg"]) if trend else "데이터 없음")

    kw = "{} {}".format(product["artist_name"], product["product_name"])
    news = get_news(kw, START_DATE, END_DATE)
    print("  뉴스 {}건".format(len(news)))

    cafe = get_cafe(kw, START_DATE, END_DATE)
    print("  카페글 {}건".format(len(cafe)))

    print("  Claude 분석 중...")
    ai_text = analyze(product, trend, news, cafe)

    results.append({"product": product, "trend": trend, "news": news, "cafe": cafe, "analysis": ai_text})
    print("  완료")

print("\n분석 완료!")

In [ ]:
# ── HTML 리포트 생성 & 저장 ──────────────────────────────────

def make_arrow(pct):
    if pct > 5:  return '<span style="color:#16a34a;font-weight:700">▲ {}%</span>'.format(abs(pct))
    if pct < -5: return '<span style="color:#dc2626;font-weight:700">▼ {}%</span>'.format(abs(pct))
    return '<span style="color:#64748b;font-weight:700">― {}%</span>'.format(abs(pct))

def make_spark(daily):
    if not daily: return ""
    vals = [d["ratio"] for d in daily]
    mn, mx = min(vals), max(vals)
    rng = mx - mn or 1
    W, H = 100, 28
    pts = ["{:.1f},{:.1f}".format(i/(max(len(vals)-1,1))*W, H-((v-mn)/rng*(H-4)+2)) for i,v in enumerate(vals)]
    return '<svg width="{}" height="{}" viewBox="0 0 {} {}"><polyline points="{}" fill="none" stroke="#6366f1" stroke-width="1.5" stroke-linejoin="round"/></svg>'.format(W,H,W,H," ".join(pts))

cards = ""
for rank, r in enumerate(results, 1):
    p, tr, nws, caf, ai = r["product"], r["trend"], r["news"], r["cafe"], r["analysis"]
    conv = round(p["buyer_cnt"]/p["visitor_cnt"]*100,1) if p["visitor_cnt"] > 0 else 0

    trend_block = "<div style='color:#94a3b8;font-size:12px'>데이터 없음</div>"
    if tr:
        trend_block = """
        <div style='background:#f8faff;border:1px solid #e0e7ff;border-radius:8px;padding:10px 14px;display:flex;align-items:center;justify-content:space-between'>
          <div>
            <div style='font-size:11px;color:#64748b;margin-bottom:4px'>네이버 검색량 지수</div>
            <div style='font-size:13px'>이번주 <b>{}</b> vs 전주 <b>{}</b> &nbsp;{}</div>
          </div>
          <div>{}</div>
        </div>""".format(tr["this_avg"], tr["prev_avg"], make_arrow(tr["change_pct"]), make_spark(tr.get("daily",[])))

    news_block = "<div style='color:#94a3b8;font-size:12px'>관련 뉴스 없음</div>"
    if nws:
        news_block = "<ul style='list-style:none;padding:0;margin:0'>" + "".join([
            "<li style='padding:4px 0;border-bottom:1px solid #f8fafc;font-size:12px'>"
            "<a href='{}' target='_blank' style='color:#3b4ff8;text-decoration:none'>{}</a>"
            " <span style='color:#94a3b8;font-size:10px'>{}</span></li>".format(n["link"],n["title"],n["pubDate"])
            for n in nws]) + "</ul>"

    cafe_block = "<div style='color:#94a3b8;font-size:12px'>카페 글 없음</div>"
    if caf:
        cafe_block = "<ul style='list-style:none;padding:0;margin:0'>" + "".join([
            "<li style='padding:4px 0;border-bottom:1px solid #f8fafc;font-size:12px'>"
            "<a href='{}' target='_blank' style='color:#7c3aed;text-decoration:none'>{}</a>"
            " <span style='color:#94a3b8;font-size:10px'>{} · {}</span></li>".format(c["link"],c["title"],c["cafeName"],c["pubDate"])
            for c in caf]) + "</ul>"

    ai_html = re.sub(r"\*\*(.+?)\*\*", r"<strong style='color:#4f46e5'>\1</strong>", ai).replace("\n","<br>")

    cards += """
    <div style='background:#fff;border-radius:14px;border:1px solid #e2e8f0;margin-bottom:20px;overflow:hidden'>
      <div style='background:#f8fafc;padding:14px 20px;border-bottom:1px solid #e2e8f0;display:flex;align-items:center;justify-content:space-between;flex-wrap:wrap;gap:10px'>
        <div>
          <span style='font-size:11px;font-weight:800;color:#94a3b8'>#{}</span>
          <span style='font-size:16px;font-weight:700;color:#0f172a;margin-left:8px'>{}</span>
          <span style='font-size:12px;color:#64748b;margin-left:6px'>{}</span>
        </div>
        <div style='display:flex;gap:20px'>
          <div style='text-align:center'><div style='font-size:15px;font-weight:700'>{:,}</div><div style='font-size:10px;color:#94a3b8'>방문자</div></div>
          <div style='text-align:center'><div style='font-size:15px;font-weight:700'>{:,}</div><div style='font-size:10px;color:#94a3b8'>구매자</div></div>
          <div style='text-align:center'><div style='font-size:15px;font-weight:700'>{}%</div><div style='font-size:10px;color:#94a3b8'>전환율</div></div>
          <div style='text-align:center'><div style='font-size:15px;font-weight:700'>₩{:,}만</div><div style='font-size:10px;color:#94a3b8'>거래액</div></div>
        </div>
      </div>
      <div style='display:flex'>
        <div style='flex:1;padding:16px 20px;min-width:0'>
          <div style='font-size:10px;font-weight:700;letter-spacing:1px;text-transform:uppercase;color:#94a3b8;margin-bottom:8px'>📈 검색 트렌드</div>
          {}
          <div style='font-size:10px;font-weight:700;letter-spacing:1px;text-transform:uppercase;color:#94a3b8;margin:14px 0 8px'>📰 뉴스</div>
          {}
          <div style='font-size:10px;font-weight:700;letter-spacing:1px;text-transform:uppercase;color:#94a3b8;margin:14px 0 8px'>☕ 카페글</div>
          {}
        </div>
        <div style='flex:0 0 320px;padding:16px 20px;background:#fafbff;border-left:1px solid #f1f5f9'>
          <div style='font-size:10px;font-weight:700;letter-spacing:1px;text-transform:uppercase;color:#94a3b8;margin-bottom:8px'>🤖 AI 해석</div>
          <div style='font-size:12.5px;line-height:1.8;color:#334155;background:#fff;border:1px solid #e0e7ff;border-radius:8px;padding:12px 14px'>{}</div>
        </div>
      </div>
    </div>""".format(
        rank, p["product_name"], p["artist_name"],
        p["visitor_cnt"], p["buyer_cnt"], conv, p["gmv"]//10000,
        trend_block, news_block, cafe_block, ai_html
    )

html = """<!DOCTYPE html><html lang='ko'><head><meta charset='UTF-8'>
<title>주간 트렌드 리포트 {s} ~ {e}</title></head>
<body style='font-family:Apple SD Gothic Neo,Malgun Gothic,Arial,sans-serif;background:#f1f5f9;padding:32px 24px;font-size:13px;color:#1e293b'>
<div style='margin-bottom:24px'>
  <div style='font-size:20px;font-weight:700'>📊 주간 실적 트렌드 리포트</div>
  <div style='font-size:12px;color:#64748b;margin-top:4px'>{s} ~ {e} &nbsp;|&nbsp; TOP {n}개 상품 (방문자·구매자 가중 점수 기준)</div>
</div>
{cards}
<div style='text-align:right;font-size:11px;color:#94a3b8;margin-top:8px'>Claude API + Naver DataLab/News/Cafe</div>
</body></html>""".format(s=START_DATE, e=END_DATE, n=len(results), cards=cards)

filename = "report_{}_{}.html".format(START_DATE, END_DATE)
with open(filename, "w", encoding="utf-8") as f:
    f.write(html)

print("리포트 저장 완료:", filename)